This notebook validates the raw landing file before building the ingestion pipeline. It checks that the source file is available in the Unity Catalog volume, confirms the dataset size and structure, and verifies that the Germany-related columns needed for downstream processing are present. This helps catch access or schema issues early before creating Bronze tables.

Set the source paths for the landing volume and the raw CSV file.

In [0]:
from pyspark.sql import functions as f

In [0]:
%run ../00_setup/03-project-config

In [0]:
"""
volume_path = "/Volumes/german_energy_transition_lakehouse/landing/raw_files"
source_file_path = f"{volume_path}/time_series_60min_singleindex.csv"

display(dbutils.fs.ls(volume_path))"""


Read the source CSV file into a Spark DataFrame and review the basic dataset size.

In [0]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(source_file_path)
)

print("Row count:", raw_df.count())
print("Column count:", len(raw_df.columns))

Identify the Germany-related columns available in the source file.

In [0]:
# germany_columns = [col for col in raw_df.columns if col.startswith("DE_")]
germany_columns = []

for column_name in raw_df.columns:
    if column_name.startswith("DE_"):
        germany_columns.append(column_name)

print("germany-related column count:", len(germany_columns))
print(germany_columns[:30])


Check whether the required columns for the Germany energy dataset are present.

In [0]:
available_required_columns = []
missing_required_columns = []

for column_name in germany_source_columns:
    if column_name in raw_df.columns:
        available_required_columns.append(column_name)
    else:
        missing_required_columns.append(column_name)

print("available required columns:", available_required_columns)
print("missing required columns:", missing_required_columns)


The day-ahead price column name needed to be confirmed. This step searches the source schema for the correct Germany price column before using it in the required column list.

In [0]:
for col in raw_df.columns:
    if "price" in col.lower() and col.startswith("DE"):
        print(col)

In [0]:
display(raw_df.select(*available_required_columns).limit(10))